# NB01 — Introducción a redes neuronales
**Correspondencia: Semanas 3–4**


## 1. Preparación del entorno


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow:", tf.__version__)


## 2. Dataset para el primer modelo

Trabajaremos con un problema de clasificación binaria sencillo. El objetivo es concentrarnos en el flujo fundamental de una red neuronal: **datos → modelo → entrenamiento → evaluación → predicción**.


In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

X, y = make_moons(
    n_samples=1000,
    noise=0.20,
    random_state=42
)

print("Dimensiones de X:", X.shape)
print("Dimensiones de y:", y.shape)
print("Clases:", np.unique(y))


## 3. Exploración de los datos


In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(X[:, 0], X[:, 1], c=y, alpha=0.7)
plt.xlabel("Característica 1")
plt.ylabel("Característica 2")
plt.title("Datos disponibles para clasificación")
plt.show()


## 4. División de los datos

Separaremos los datos en conjuntos de entrenamiento y prueba.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)


## 5. Construcción de una red neuronal

La red utilizará capas densas. Cada neurona recibe entradas, aplica pesos y sesgo, y posteriormente una función de activación.


In [ ]:
model = keras.Sequential([
    layers.Input(shape=(2,)),
    layers.Dense(8, activation="relu"),
    layers.Dense(4, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

model.summary()


## 6. Compilación del modelo

Definiremos tres elementos fundamentales:

- **Función de pérdida:** mide el error del modelo.
- **Optimizador:** actualiza los pesos.
- **Métrica:** permite observar el desempeño durante entrenamiento.


In [ ]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


## 7. Entrenamiento

Durante cada época, la red procesa los datos, calcula el error y actualiza sus pesos mediante el proceso de aprendizaje.


In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_split=0.20,
    epochs=40,
    batch_size=32,
    verbose=1
)


## 8. Curvas de aprendizaje


In [ ]:
history_df = pd.DataFrame(history.history)

plt.figure(figsize=(7, 5))
plt.plot(history_df["accuracy"], label="Entrenamiento")
plt.plot(history_df["val_accuracy"], label="Validación")
plt.xlabel("Época")
plt.ylabel("Accuracy")
plt.title("Evolución del accuracy")
plt.legend()
plt.show()


In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(history_df["loss"], label="Entrenamiento")
plt.plot(history_df["val_loss"], label="Validación")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.title("Evolución de la función de pérdida")
plt.legend()
plt.show()


## 9. Evaluación con datos no utilizados para entrenamiento


In [ ]:
test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print(f"Loss en test: {test_loss:.4f}")
print(f"Accuracy en test: {test_accuracy:.4f}")


## 10. Realizar predicciones


In [ ]:
probabilidades = model.predict(X_test[:10], verbose=0).flatten()
predicciones = (probabilidades >= 0.5).astype(int)

resultados = pd.DataFrame({
    "Probabilidad": probabilidades,
    "Clase_predicha": predicciones,
    "Clase_real": y_test[:10]
})

resultados


## 11. Visualización de la clasificación aprendida


In [ ]:
x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5

xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 300),
    np.linspace(y_min, y_max, 300)
)

grid = np.c_[xx.ravel(), yy.ravel()]
zz = model.predict(grid, verbose=0).reshape(xx.shape)

plt.figure(figsize=(7, 5))
plt.contourf(xx, yy, zz, levels=[0, 0.5, 1], alpha=0.25)
plt.scatter(X_test[:, 0], X_test[:, 1], c=y_test, alpha=0.8)
plt.xlabel("Característica 1")
plt.ylabel("Característica 2")
plt.title("Frontera de decisión aprendida")
plt.show()


## 12. Experimentación

Modifique **un elemento a la vez** y vuelva a entrenar el modelo.

Pruebe, por ejemplo:

- cantidad de neuronas;
- cantidad de capas ocultas;
- función de activación;
- número de épocas;
- tamaño del batch.

Registre el cambio realizado y compare el resultado con el modelo original.


In [ ]:
# Modelo experimental
modelo_experimental = keras.Sequential([
    layers.Input(shape=(2,)),
    layers.Dense(16, activation="relu"),
    layers.Dense(8, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

modelo_experimental.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

historial_experimental = modelo_experimental.fit(
    X_train,
    y_train,
    validation_split=0.20,
    epochs=40,
    batch_size=32,
    verbose=0
)

loss_exp, acc_exp = modelo_experimental.evaluate(
    X_test,
    y_test,
    verbose=0
)

print(f"Accuracy modelo experimental: {acc_exp:.4f}")


## 13. Comparación de resultados


In [ ]:
comparacion = pd.DataFrame({
    "Modelo": ["Modelo inicial", "Modelo experimental"],
    "Accuracy_test": [test_accuracy, acc_exp]
})

comparacion


## 14. Actividad

Responda a partir de los resultados obtenidos:

1. ¿Qué función cumplen las capas ocultas de la red?
2. ¿Qué función cumple la capa de salida?
3. ¿Qué representa la función de pérdida?
4. ¿Qué diferencia observa entre entrenamiento, validación y prueba?
5. ¿El cambio realizado en el modelo experimental mejoró el desempeño?
6. ¿Qué modificación probaría a continuación y por qué?


## 15. Base para la Evaluación 1

Utilice el flujo desarrollado en este notebook como referencia para construir el **prototipo funcional inicial** del proyecto integrador:

**datos → arquitectura → compilación → entrenamiento → validación → predicción**.
